In [5]:
from operator import add
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langchain_core.messages import HumanMessage, BaseMessage
from typing import TypedDict, Annotated
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

class CustomState(TypedDict):
    messages: Annotated[list[BaseMessage], add]

def node_llm(state: CustomState):
    messages = state.get("messages", [])
    response = model.invoke(messages)
    # print("AI: ", response)
    return {"messages": [response]}

builder = StateGraph(CustomState)

builder.add_node("node_llm", node_llm)
builder.add_edge(START, "node_llm")
builder.add_edge("node_llm", END)

graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage("你好👋，你是谁")]})
print(result)

{'messages': [HumanMessage(content='你好👋，你是谁', additional_kwargs={}, response_metadata={}), AIMessage(content='你好呀！👋\n\n我是**DeepSeek**，由深度求索公司创造的AI助手！很高兴认识你～\n\n简单介绍一下我自己：\n- 🤖 我可以帮你回答问题、写作、编程、分析资料等等\n- 📚 支持上传文件（图片、PDF、Word、Excel等）并读取其中的文字信息\n- 🔗 可以阅读链接内容\n- 🌐 支持联网搜索（需要手动开启）\n- 💬 上下文容量超大（1M），能一次性处理像《三体》三部曲那么多的内容\n- 🆓 完全免费使用！\n\n我的知识截止到2026年2月，是DeepSeek最新版本模型。你可以上传图片给我，我能识别和分析图片中的可见信息。\n\n有什么我可以帮你的吗？无论是聊天、学习、工作还是创作，我都很乐意帮忙！😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 176, 'prompt_tokens': 9, 'total_tokens': 185, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '5f1cfb64-2ef8-4544-8105-dc777fb7053d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a216c1-d9b0-7320-b5ee-11d6948527dd-0', too